# 实验 03 · 逻辑回归与正则化

目标：处理数值稳定性，并确认正则化不作用于偏置。

本实验为新增练习，仅依赖 NumPy；题干、测试均不包含参考实现。
从上到下执行，完成每个 TODO 后运行对应自测。

In [2]:
import numpy as np
rng = np.random.default_rng(42)

## 1 · 稳定的 Sigmoid

实现 sigmoid_stable(z)，支持标量与任意形状数组，保留数组形状。对于 ±1000 也不能发生溢出。

In [3]:
def sigmoid_stable(z):
    is_scalar = np.isscalar(z)
    z = np.asarray(z)
    e = np.exp(-np.abs(z))
    result = np.where(z>=0, 1.0, e) / (1.0+e)
    if is_scalar:
        return result.item()
    return result

    raise NotImplementedError("TODO: 稳定的 Sigmoid")

In [4]:
with np.errstate(over='raise', invalid='raise'):
    np.testing.assert_allclose(sigmoid_stable(np.array([-1000., 0., 1000.])), [0., 0.5, 1.])
np.testing.assert_allclose(sigmoid_stable(0.), 0.5)
assert np.asarray(sigmoid_stable(np.zeros((2, 3)))).shape == (2, 3)
np.testing.assert_allclose(sigmoid_stable(np.array([-2., 2.])).sum(), 1.)
print("通过：极端输入、标量、多维形状")

通过：极端输入、标量、多维形状


## 2 · 正则化梯度

实现 logistic_gradient(X,y,w,b,lambda_)，返回 (dw,db)，形状分别 (n,) 与标量。使用平均二元交叉熵，正则项为 lambda_/(2m) × 权重平方和；不正则化 b。

In [6]:
def logistic_gradient(X, y, w, b, lambda_):

    X = np.asarray(X)
    y = np.asarray(y).ravel()
    w = np.asarray(w).ravel()
    m = X.shape[0]

    p = sigmoid_stable(X @ w + b)
    d = p-y
    dw = (X.T @ d)/m + (lambda_/m)*w
    db = float(d.mean())
    return (X.T @ d)/m + (lambda_/m)*w, d.mean()

    raise NotImplementedError("TODO: 带 L2 正则的梯度")

In [7]:
X = np.array([[1., 0.], [0., 2.]])
y = np.array([0., 1.]); w = np.array([0.4, -0.2]); b = 0.3
dw0, db0 = logistic_gradient(X, y, w, b, 0.)
dw1, db1 = logistic_gradient(X, y, w, b, 3.)
np.testing.assert_allclose(dw1 - dw0, [0.6, -0.3])
np.testing.assert_allclose(db1, db0)
d0, b0 = logistic_gradient(X, y, np.zeros(2), 0., 0.)
np.testing.assert_allclose(d0, [0.25, -0.5]); np.testing.assert_allclose(b0, 0.)
assert np.asarray(dw0).shape == (2,) and np.asarray(db0).ndim == 0
X1 = np.array([[1.], [2.]])
np.testing.assert_allclose(logistic_gradient(X1, np.ones(2), np.zeros(1), 0., 0.)[1], -0.5)
print("通过：已知值、正则项、偏置不正则化")

通过：已知值、正则项、偏置不正则化


## 实验记录

写出输入/输出形状，解释一个边界情况，再记录一次修改参数前后的观察。

`sigmoid_stable(z)`: 输入任意形状的 NumPy 数组，输出与输入同形状；`logistic_gradient(X, y, w, b, lambda_)` 输入 `X` 形状 `(m, n)`，`y` 形状 `(m,)`，`w` 形状 `(n,)`，`b` 为标量，`lambda_` 为标量，输出 `dw` 形状 `(n,)`，`db` 为标量。
 
`sigmoid_stable` 需要处理极端输入，例如 `z = ±1000`。实现中不会直接计算 `exp(1000)`，避免溢出。当 `z = -1000` 时 `e` 下溢为 0，输出为 0；当 `z = 1000` 时输出为 `1/(1+e)`，近似为 1。另一个边界是偏置 `b` 不正则化：当 `lambda_` 从 0 改为 3 时，`db` 保持不变，而 `dw` 增加 `(lambda_/m) * w`。

取测试数据：

```python
X = np.array([[1., 0.], [0., 2.]])
y = np.array([0., 1.])
w = np.array([0.4, -0.2])
b = 0.3
```


当 `lambda_ = 0` 时得到 `(dw0, db0)`；当 `lambda_ = 3` 时得到 `(dw1, db1)`。观察到：

```python
dw1 - dw0 = [0.6, -0.3]
db1 == db0
```

这是因为 `m = 2`，L2 正则项为 `lambda_/(2m) * ||w||^2`，其对 `w` 的梯度为 `(lambda_/m) * w`：

```python
(3 / 2) * [0.4, -0.2] = [0.6, -0.3]
```

而正则项不包含 `b`，所以 `b` 的梯度不受 `lambda_` 影响。另在 `w = 0, b = 0, lambda_ = 0` 时，得到 `d0 = [0.25, -0.5]`、`b0 = 0`，与平均二元交叉熵梯度的手算结果一致。
